#  Multi-Agents: Supervisor Agent 만들기

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings("ignore")

---

## 🎯 Supervisor 패턴이란?

- 감독자(supervisor) 에이전트가 여러 전문 작업자(worker) 에이전트를 조정하는 구조

- 하나의 에이전트에 모든 도구를 제공하는 대신:
    - 📅 **달력 관리 전문가**: 일정 조회, 이벤트 생성
    - 📧 **이메일 관리 전문가**: 메일 작성, 발송

- 각 전문가가 자신의 영역에 집중하고, Supervisor가 이들을 조율하는 구조


## 1. 기본 도구(Tools) 정의하기

In [ ]:
from langchain_core.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,  # 예: "2024-01-15T14:00:00"
    end_time: str,
    attendees: list[str]
) -> str:
    """달력에 이벤트를 생성합니다."""
    return f"✅ 이벤트 생성됨: {title} ({start_time}~{end_time}), 참석자: {len(attendees)}명"


@tool
def send_email(
    to: list[str],
    subject: str,
    body: str
) -> str:
    """이메일을 발송합니다."""
    return f"✅ 이메일 발송 완료 - 수신: {', '.join(to)}, 제목: {subject}"


@tool
def get_available_time_slots(
    date: str,  # 예: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """특정 날짜의 가능한 시간대를 조회합니다."""
    return ["09:00", "14:00", "16:00"]

## 2. 전문 에이전트 만들기

In [ ]:
### 달력 에이전트 생성

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 모델 초기화
model = init_chat_model("gpt-4.1-mini")  

# 달력 전문 에이전트
calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=(
        "당신은 일정 관리 비서입니다. "
        "자연어 요청(예: '다음 화요일 오후 2시')을 "
        "ISO 날짜 형식으로 변환하고 일정을 생성하세요."
    )
)

# 테스트해보기
query = "내일 오전 9시에 팀 회의를 잡아줘"

for step in calendar_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            print(f"{message.type}: {message.content}")

In [ ]:
### 이메일 에이전트 생성

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=(
        "당신은 이메일 작성 비서입니다. "
        "자연어 요청을 전문적인 이메일로 변환하여 발송하세요."
    )
)

# 테스트해보기
query = "디자인팀에게 새 목업 검토 요청 메일을 보내줘"

for step in email_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            print(f"{message.type}: {message.content}")

## 3. 에이전트를 도구로 감싸기

- Supervisor가 사용할 수 있도록 각 에이전트를 도구로 변환

In [ ]:
@tool
def schedule_event(request: str) -> str:
    """
    자연어로 일정을 예약합니다.
    
    사용 예: '다음 주 화요일 오후 2시에 디자인팀 미팅'
    """
    result = calendar_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


@tool
def manage_email(request: str) -> str:
    """
    자연어로 이메일을 발송합니다.
    
    사용 예: '그들에게 회의 리마인더를 보내줘'
    """
    result = email_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content

## 4. Supervisor 에이전트 만들기

In [ ]:
supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=(
        "당신은 유능한 개인 비서입니다. "
        "일정 관리와 이메일 발송을 할 수 있습니다. "
        "사용자 요청을 적절한 도구로 분해하고 결과를 조율하세요."
    )
)

## 5. 완성된 시스템 사용하기

In [ ]:
### 예제 1: 단일 작업

query = "내일 오전 9시에 팀 스탠드업 회의를 잡아줘"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

In [ ]:
### 예제 2: 복합 작업

query = (
    "다음 주 화요일 오후 2시에 디자인팀과 1시간 회의를 잡고, "
    "새 목업 검토에 대한 리마인더 이메일도 보내줘"
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

## 6. Human-in-the-loop 추가하기

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 달력 에이전트에 승인 절차 추가
CALENDAR_AGENT_PROMPT = (
    "당신은 일정 관리 비서입니다. "
    "자연어 요청(예: '다음 화요일 오후 2시')을 "
    "ISO 날짜 형식으로 변환하고 일정을 생성하세요."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"create_calendar_event": True},  # 이 도구 실행 시 중단
            description_prefix="📅 달력 이벤트 승인 대기 중",
        ),
    ],
)

# 이메일 에이전트에 승인 절차 추가
EMAIL_AGENT_PROMPT = (
    "당신은 이메일 작성 비서입니다. "
    "자연어 요청을 전문적인 이메일로 변환하여 발송하세요."
)
email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},  # 이 도구 실행 시 중단
            description_prefix="📧 이메일 발송 승인 대기 중",
        ),
    ],
)

# Supervisor에 체크포인터 추가 (실행 상태 저장용)
SUPERVISOR_PROMPT = (
    "당신은 유능한 개인 비서입니다. "
    "일정 관리와 이메일 발송을 할 수 있습니다. "
    "사용자 요청을 적절한 도구로 분해하고 결과를 조율하세요. "
    "여러 작업이 필요한 경우 순차적으로 도구를 사용하세요."
)
supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),  # 메모리에 상태 저장
)

In [ ]:
### 승인이 필요한 작업 실행하기

query = (
    "다음 주 화요일 오후 2시에 디자인팀과 1시간 회의를 잡고, "
    "새 목업 검토에 대한 리마인더 이메일도 보내줘"
)

# 스레드 ID로 대화 추적
config = {"configurable": {"thread_id": "conversation_1"}}

# 중단 이벤트를 저장할 리스트
interrupts = []

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            # 일반 메시지 출력
            for message in update.get("messages", []):
                message.pretty_print()
        else:
            # 중단 이벤트 저장
            interrupt_ = update[0]
            interrupts.append(interrupt_)
            print(f"\n⚠️ 중단됨: {interrupt_.id}")

In [ ]:
### 중단된 작업 확인하기

for interrupt_ in interrupts:
    for request in interrupt_.value["action_requests"]:
        print(f"\n🔔 중단 ID: {interrupt_.id}")
        print(f"📋 {request['description']}\n")

In [ ]:
from langgraph.types import Command

# 승인/수정/거부 결정하기
resume = {}

for interrupt_ in interrupts:
    print(f"\n처리 중: {interrupt_.id}")
    
    # action_requests 확인
    action_requests = interrupt_.value.get("action_requests", [])
    
    if not action_requests:
        print("⚠️ action_requests가 비어있습니다.")
        continue
    
    request = action_requests[0]
    
    # 이메일 발송 중단인 경우 제목 수정
    if interrupt_.id == "b276f6620010cc8f09ec4d5c6b45bbf1":
        # 올바른 방법: 전체 request 복사 후 수정
        edited_action = request.copy()
        
        # args 또는 arguments 확인 후 수정
        if "args" in edited_action:
            edited_action["args"]["subject"] = "목업 검토 요청"
        elif "arguments" in edited_action:
            edited_action["arguments"]["subject"] = "목업 검토 요청"
        else:
            # 구조를 모르는 경우 전체 출력
            print(f"⚠️ 알 수 없는 구조: {edited_action}")
            continue
        
        resume[interrupt_.id] = {
            "decisions": [{"type": "edit", "edited_action": edited_action}]
        }
        print(f"✏️ 이메일 제목 수정됨")
    else:
        # 다른 작업은 승인
        resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}
        print(f"✅ 승인됨")

# 재개
for step in supervisor_agent.stream(
    Command(resume=resume),
    config,
):
    for update in step.values():
        if isinstance(update, dict):
            for message in update.get("messages", []):
                message.pretty_print()